# Access Share Point Online (SPO)


from .pfx to .pem
`openssl pkcs12 -legacy -info -in protected.pfx -nodes > bosnet.crt`

### Documentation on how to install outdated crypto libraries on MacOS
```
brew install openssl@1.1 
ln /opt/homebrew/anaconda3/envs/py38/lib/libssl.1.1.dylib /opt/homebrew/anaconda3/lib/.
ln /opt/homebrew/anaconda3/envs/py38/lib/libcrypto.1.1.dylib /opt/homebrew/anaconda3/lib/.
```

## Dependencies and logging

In [ ]:
import sys
import cryptography

In [ ]:
import logging

logging.getLogger().setLevel(logging.DEBUG)
requests_log = logging.getLogger("requests.packages.urllib3")
requests_log.setLevel(logging.DEBUG)
requests_log.propagate = True

In [ ]:
from requests import help as python_requests_library_help
python_requests_library_help.info()

# Configuration

In [ ]:
sys.path.append('./lib/Office365-REST-Python-Client')

from office365.sharepoint.client_context import ClientContext
from office365.runtime.auth.user_credential import UserCredential
from office365.runtime.auth.authentication_context import AuthenticationContext


app_settings_sandbox = {
    'url': 'https://bosnet.sharepoint.com/sites/Bossard_MasterDataSandbox',
    'tenant': '4ee0b6db-2682-4f0d-8356-b8b71d6af335',
    'client_id': '6f708579-76a6-4009-8f96-f35c234239e1', 
    'thumbprint': '54986C04948A0B724C8608B41F5F35372A4E97CB',
    'certificate_path': 'bosnet.crt',
}

# Should fail if permissions are restrictive. 'Entities' list exists.
app_settings_production = {
    'url': 'https://bosnet.sharepoint.com/sites/BossardMasterData',
    'tenant': '4ee0b6db-2682-4f0d-8356-b8b71d6af335',
    'client_id': '6f708579-76a6-4009-8f96-f35c234239e1', 
    'thumbprint': '54986C04948A0B724C8608B41F5F35372A4E97CB',
    'certificate_path': 'bosnet.crt',
}

# Should fail as 'Entities' list does not exist
app_settings_it = {
    'url': 'https://bosnet.sharepoint.com/sites/Group_IT',
    'tenant': '4ee0b6db-2682-4f0d-8356-b8b71d6af335',
    'client_id': '6f708579-76a6-4009-8f96-f35c234239e1', 
    'thumbprint': '54986C04948A0B724C8608B41F5F35372A4E97CB',
    'certificate_path': 'bosnet.crt',
}

## Choose which environment you want to use

In [ ]:
# app_settings = app_settings_production
# app_settings = app_settings_it
app_settings = app_settings_sandbox

logging.debug(f"Using {app_settings['url']}")

In [ ]:
ctx = ClientContext.from_url(app_settings['url']).with_client_certificate(app_settings['tenant'], 
                                     app_settings['client_id'], 
                                     app_settings['thumbprint'],
                                     app_settings['certificate_path'])

In [ ]:
id = ctx.site.get().execute_query()
print(id)

In [ ]:
entity_list = ctx.web.lists.get_by_title("Entities")

In [ ]:
items = entity_list.items.paged(False).top(1000)
result = items.get_items_count().execute_query()
logging.info(f"Total items count: {result.value}")

In [ ]:
content = items.get().execute_query()

In [ ]:
element0 = content[0]
element0.__dict__

# Try to add a list element

In [ ]:
import copy
element1 = copy.deepcopy(element0._properties)
element1.pop('Id')
element1.pop('ID')
element1.pop('GUID')
element1['Title'] = 'New element ' + str(int(result.value) + 1)

In [ ]:
entity_list.add_item(element1)
entity_list.execute_query()